# protocol

> Claude Code's stream-json wire protocol: NDJSON transport, control routing, and the callable-tool bridge

In [ ]:
#| default_exp protocol

Speak Claude Code's stream-json wire protocol directly, with no Agent SDK: `read_msgs` frames NDJSON from a piped process, `mk_tools` turns annotated callables or `(schema, callable)` pairs into an advertised tool list and a dispatch namespace, `mcp_dispatch` implements the five MCP-shaped JSON-RPC methods the CLI uses for in-process tools, and `ClaudeProto` is the per-process peer: it matches `control_response`s to pending requests, runs each incoming `control_request` as its own cancellable task, answers `control_cancel_request` by cancelling and staying silent, and yields every other message through untouched. `initialize` and `interrupt` are ordinary control requests. The runner in `fastclaude.core` owns the process itself: command line, environment, transcript, and cleanup.

In [ ]:
#| export
import asyncio, json, os
from fastcore.utils import *
from fastcore.funccall import get_schema, call_func, call_func_async

In [ ]:
from fastcore.test import *
from fastclaude.session import ant_data, sess_dir
from collections import Counter
import shutil, sys, tempfile, textwrap

## The stream

Run with `--output-format stream-json --verbose --include-partial-messages` and piped stdio, `claude` becomes a headless NDJSON peer: every line of stdout is one JSON message, and stdin accepts JSON messages the same way. The fixture below is one real captured run: a Bash tool call, made against a scratch project. Like the transcript fixtures in `fastclaude.session`, the capture is checked in as package data, and the builder returns immediately when it exists; delete the file to request a fresh capture against the installed CLI.

In [ ]:
stream_path = ant_data.parent/'stream.jsonl'

In [ ]:
async def mk_stream_fixture(path=None):  # chkstyle: ignore-node
    "Capture one live claude run's raw stream-json stdout as the checked-in stream fixture"
    path = Path(path) if path else stream_path
    if path.exists(): return path
    prompt = 'think hard: first work out 17*23-4 in your head, then use the Bash tool to run: echo flux-41.7 . Then reply with exactly the tool output followed by your arithmetic answer.'
    argv = ['claude','-p','--output-format','stream-json','--verbose','--include-partial-messages','--model','sonnet','--allowedTools','Bash']
    td = tempfile.mkdtemp()
    p = await asyncio.create_subprocess_exec(*argv, stdin=asyncio.subprocess.PIPE, stdout=asyncio.subprocess.PIPE, limit=2**25, cwd=td)
    out,_ = await p.communicate(prompt.encode())
    assert not p.returncode, f'claude exited {p.returncode}'
    path.write_bytes(out)
    shutil.rmtree(sess_dir(td), ignore_errors=True)
    return path

In [ ]:
await mk_stream_fixture()

A run's stdout mixes several kinds of message. Counting them first shows the shape of a whole conversation before we pick each kind apart:

In [ ]:
evs = dict2obj(stream_path.read_jsonl())
len(evs),Counter(e.type for e in evs)

The same content arrives twice. As the model generates, `stream_event` messages wrap the raw Anthropic SSE stream (`message_start`, `content_block_delta`, ...), token by token, for streaming consumers. Then, as each content block completes, a full `assistant` message event carries the finished block; tool results arrive as full `user` message events. The full events match what Claude writes to the session transcript, record for record (same `uuid`s, same `message` content), so a consumer that wants the finished conversation reads only them and ignores the partials:

In [ ]:
tu_ev = first(e for e in evs if e.type=='assistant' and e.message.content[0].type=='tool_use')
tr_ev = first(e for e in evs if e.type=='user')
test_eq(tr_ev.message.content[0].tool_use_id, tu_ev.message.content[0].id)
deltas = [d.event.delta for d in evs if d.type=='stream_event' and d.event.type=='content_block_delta']
test_eq(json.loads(''.join(d.partial_json for d in deltas if d.type=='input_json_delta')), obj2dict(tu_ev.message.content[0].input))
tu_ev.message.content[0].name, tr_ev.message.content[0].content

The rest is ambient noise a reader must tolerate rather than parse: `system` events (`init` with the session's tools and config, `status`, `thinking_tokens`, and `hook_started`/`hook_response` when the user's own hooks fire, which they do even headless), `rate_limit_event`, and whatever new types later CLI versions add. The protocol layer therefore never enumerates event types: it routes the three control messages it owns and passes everything else through untouched.

## NDJSON framing

Pipes deliver chunks, not lines: one read can return half a message, or three. asyncio's `StreamReader.readline` owns the reassembly, bounded by the `limit` passed when the process is spawned (one claude message can carry megabytes of base64 image, so the runner spawns with a generous limit rather than the 64KB default). `read_msgs` turns the byte stream into decoded messages, skipping blank lines and failing loudly on anything that is not JSON: silently dropping a line would hide a desynced stream.

In [ ]:
#| export
async def read_msgs(
    stream, # An asyncio `StreamReader` of NDJSON, e.g. a claude process's stdout
):
    "Decoded messages from `stream`, one per line; blank lines skip, non-JSON raises, a truncated final fragment drops"
    while line := await stream.readline():
        if not (s := line.strip()): continue
        try: yield json.loads(s)
        except json.JSONDecodeError as e:
            if line.endswith(b'\n'): raise ValueError(f'bad NDJSON line: {s[:200]!r}') from e
            return  # no newline: a producer killed mid-write; the fragment is unrecoverable

A scripted producer replays the captured stream in aggressive 37-byte writes, so nearly every message arrives split across reads. Framing reassembles exactly the events the capture holds:

In [ ]:
emit = f"""import sys
data = open({str(stream_path)!r}, 'rb').read()
for i in range(0, len(data), 37):
    sys.stdout.buffer.write(data[i:i+37])
    sys.stdout.buffer.flush()"""
p = await asyncio.create_subprocess_exec(sys.executable, '-c', emit, stdout=asyncio.subprocess.PIPE, limit=2**25)
got = [m async for m in read_msgs(p.stdout)]
await p.wait()
test_eq(got, list(obj2dict(evs)))
test_eq(got[-1]['type'], 'result')

Blank lines skip, a complete final line missing only its newline still arrives, a fragment truncated mid-write is dropped (only a killed producer leaves one), and a non-JSON line raises rather than desyncing:

In [ ]:
def feedr(b):
    "A `StreamReader` pre-fed with `b`, at EOF"
    r = asyncio.StreamReader()
    r.feed_data(b)
    r.feed_eof()
    return r

test_eq([m async for m in read_msgs(feedr(b'{"a": 1}\n\n{"b": 2}'))], [dict(a=1), dict(b=2)])
test_eq([m async for m in read_msgs(feedr(b'{"a": 1}\n{"cut": tru'))], [dict(a=1)])
bad = read_msgs(feedr(b'not json\n'))
with expect_fail(ValueError, contains='bad NDJSON'): await bad.__anext__()

## Callable tools

A public tool is an ordinary annotated Python callable with a docstring: `get_schema` derives its name, description, and JSON schema from the signature, so there is no registry and no wrapper class. Hosts whose tools live elsewhere (ipyai and solveit fetch schemas from a kernel and dispatch by name) pass an explicit `(schema, callable)` pair instead, where the schema dict carries `name`, `description`, and `inputSchema`. `mk_tools` normalizes a mixed list of both forms into the two things the bridge needs: the schema list to advertise, and the namespace to dispatch into.

In [ ]:
#| export
def tool_spec(
    t, # An annotated callable, or an explicit `(schema, callable)` pair
):
    "`(schema, callable)` for one tool; a callable's schema is derived from its signature"
    if callable(t): return get_schema(t, pname='inputSchema'), t
    return t

def mk_tools(
    tools, # Tools in either `tool_spec` form
):
    "`(schemas, ns)` for `tools`: the schema list to advertise, and the dispatch namespace"
    specs = [tool_spec(t) for t in listify(tools)]
    return [s for s,_ in specs], {s['name']: f for s,f in specs}

In [ ]:
async def flux_meter(unit:str='kf') -> str:
    "Read the flux."
    return f'flux: 41.7 {unit}'

py_schema = dict(name='py', description='Run code in the kernel',
    inputSchema=dict(type='object', properties=dict(code=dict(type='string')), required=['code']))
async def _py_caller(**kw): return f"ran: {kw['code']}"

schemas,ns = mk_tools([flux_meter, (py_schema, _py_caller)])
test_eq([s['name'] for s in schemas], ['flux_meter','py'])
test_eq(set(ns), {'flux_meter','py'})
schemas[0]

## The MCP-shaped bridge

Claude Code talks to in-process tools in MCP's JSON-RPC shapes, nested inside its control protocol (the next section). The dispatcher below implements exactly the methods the CLI uses for callable tools - `initialize`, `notifications/*`, `ping`, `tools/list`, `tools/call` - and nothing else. It is deliberately not a general MCP implementation, and `fastclaude` does not depend on the `mcp` package.

A tool's return value becomes MCP `content`: a string becomes text, `None` an empty success, and any other JSON-compatible value readable JSON text. An exception inside a tool becomes an `isError` result Claude can read and react to, never a protocol error; protocol errors are reserved for JSON-RPC itself, such as an unknown method.

In [ ]:
#| export
def tool_content(
    v, # A callable tool's return value
):
    "MCP `content` list for `v`: text for a str, empty for None, readable JSON otherwise"
    if v is None: return []
    if not isinstance(v, str): v = json.dumps(v, ensure_ascii=False, default=str)
    return [dict(type='text', text=v)]

In [ ]:
#| export
async def mcp_dispatch(
    msg, # One JSON-RPC message from the CLI
    schemas, # Tool schemas to advertise, from `mk_tools`
    ns, # Dispatch namespace, from `mk_tools`
    server='fastclaude', # Server name reported to the CLI
):
    "The JSON-RPC response for `msg`, or None for a notification; a sync tool runs in a worker thread"
    m,i = msg.get('method'), msg.get('id')
    def res(r): return dict(jsonrpc='2.0', id=i, result=r)
    if m == 'initialize':
        pv = nested_idx(msg, 'params', 'protocolVersion') or '2024-11-05'
        return res(dict(protocolVersion=pv, capabilities=dict(tools={}), serverInfo=dict(name=server, version='1.0')))
    if m and m.startswith('notifications/'): return None
    if m == 'ping': return res({})
    if m == 'tools/list': return res(dict(tools=schemas))
    if m == 'tools/call':
        nm,args = nested_idx(msg, 'params', 'name'), nested_idx(msg, 'params', 'arguments') or {}
        try:
            if asyncio.iscoroutinefunction(ns.get(nm)): v = await call_func_async(nm, args, ns)
            else: v = await asyncio.to_thread(call_func, nm, args, ns)
            return res(dict(content=tool_content(v), isError=False))
        except asyncio.CancelledError: raise
        except Exception as e: return res(dict(content=[dict(type='text', text=f'{type(e).__name__}: {e}')], isError=True))
    return dict(jsonrpc='2.0', id=i, error=dict(code=-32601, message=f'method not found: {m}'))

The dispatcher is a pure function of one message, so its whole contract can be read off in-memory calls. The handshake echoes the client's protocol version, a notification returns None (the outer control request still gets acknowledged, in the next section), and the tool listing is the schemas exactly as built:

In [ ]:
r = await mcp_dispatch(dict(jsonrpc='2.0', id=1, method='initialize', params=dict(protocolVersion='2025-06-18')), schemas, ns)
test_eq(r['result']['protocolVersion'], '2025-06-18')
test_eq(r['result']['serverInfo']['name'], 'fastclaude')
test_is(await mcp_dispatch(dict(jsonrpc='2.0', method='notifications/initialized'), schemas, ns), None)
test_eq((await mcp_dispatch(dict(jsonrpc='2.0', id=2, method='ping'), schemas, ns))['result'], {})
test_eq((await mcp_dispatch(dict(jsonrpc='2.0', id=3, method='tools/list'), schemas, ns))['result']['tools'], schemas)
r

`tools/call` runs the callable and shapes its result. The async tool defined above answers directly; the kernel-style pair dispatches by name with its keyword arguments; and a plain sync function runs in a worker thread, so a slow one cannot block the event loop that must keep answering Claude's control traffic:

In [ ]:
def tcall(nm, **args): return dict(jsonrpc='2.0', id=9, method='tools/call', params=dict(name=nm, arguments=args))

r = await mcp_dispatch(tcall('flux_meter', unit='gauss'), schemas, ns)
test_eq(r['result'], dict(content=[dict(type='text', text='flux: 41.7 gauss')], isError=False))
r = await mcp_dispatch(tcall('py', code='6*7'), schemas, ns)
test_eq(r['result']['content'][0]['text'], 'ran: 6*7')

A plain sync function needs no wrapper either; its return value serializes as readable JSON, and the worker thread keeps the loop free:

In [ ]:
def sync_add(a:int, b:int) -> int:
    "Add two numbers."
    return a+b
s2,n2 = mk_tools([sync_add])
r = await mcp_dispatch(tcall('sync_add', a=3, b=4), s2, n2)
test_eq(r['result']['content'], [dict(type='text', text='7')])

An exception inside a tool, including an unknown tool name, comes back as an `isError` result naming the exception, so Claude can read what went wrong and try again differently. Only JSON-RPC itself gets a protocol error, with the standard method-not-found code:

In [ ]:
def fussy():
    "Always refuses."
    raise ValueError('no flux today')
s3,n3 = mk_tools([fussy])
r = await mcp_dispatch(tcall('fussy'), s3, n3)
test_eq(r['result']['isError'], True)
test_eq(r['result']['content'][0]['text'], 'ValueError: no flux today')

An unknown tool name is a tool error too, since Claude chose the name and can correct itself; an unknown JSON-RPC method is the protocol's own error:

In [ ]:
r = await mcp_dispatch(tcall('nosuch'), schemas, ns)
test_eq(r['result']['isError'], True)
r = await mcp_dispatch(dict(jsonrpc='2.0', id=4, method='resources/list'), schemas, ns)
test_eq(r['error']['code'], -32601)

## Control routing

Control traffic rides the same NDJSON stream as everything else, as three message types. A `control_request` carries a `request_id` and a `request` dict, in both directions: Claude sends one to reach our tools (`subtype: mcp_message`, wrapping one JSON-RPC message), and we send one for the handshake (`subtype: initialize`) or a native interrupt. A `control_response` answers one by id. A `control_cancel_request` tells us Claude has abandoned a request it made: the handler is cancelled, and no response is written for it.

`ClaudeProto` is the peer for one process: it matches responses to pending requests, spawns one tracked task per incoming request so a slow tool call never blocks the stream, and yields every non-control message through untouched. A JSON-RPC notification returns nothing inner, but the outer control request still gets its acknowledgement, or Claude would wait on it forever.

In [ ]:
#| export
class ClaudeProto:
    "Control-protocol peer for one claude process: request matching, tool routing, passthrough events"
    def __init__(self,
        proc, # An asyncio subprocess speaking stream-json on piped stdin/stdout
        tools=None, # Tools in either `tool_spec` form
        server='fastclaude', # SDK MCP server name, matching the `--mcp-config` entry
    ):
        self.proc,self.server = proc,server
        self.schemas,self.ns = mk_tools(tools or [])
        self._lock,self._n,self._pending,self._inflight = asyncio.Lock(),0,{},{}

    async def send(self, obj):
        "Write one JSON message to claude's stdin"
        async with self._lock:
            self.proc.stdin.write(json.dumps(obj, ensure_ascii=False).encode()+b'\n')
            await self.proc.stdin.drain()

    async def send_req(self, req, timeout=60):
        "Send a control request, await its response by id, and return the inner `response` dict"
        self._n += 1
        rid = f'req_{self._n}_{os.urandom(4).hex()}'
        fut = asyncio.get_running_loop().create_future()
        self._pending[rid] = fut
        await self.send(dict(type='control_request', request_id=rid, request=req))
        try: return await asyncio.wait_for(fut, timeout)
        finally: self._pending.pop(rid, None)

    async def initialize(self, timeout=120):
        "The control-protocol handshake; claude answers nothing else until it completes"
        return await self.send_req(dict(subtype='initialize', hooks=None), timeout)

    async def interrupt(self, timeout=30):
        "Claude's native interrupt: end the current turn, keeping the process alive"
        return await self.send_req(dict(subtype='interrupt'), timeout)

In [ ]:
#| export
@patch
def _resolve(self:ClaudeProto, msg):
    "Complete the pending request a `control_response` answers"
    r = msg.get('response') or {}
    if (fut := self._pending.get(r.get('request_id'))) and not fut.done():
        if r.get('subtype')=='error': fut.set_exception(RuntimeError(r.get('error') or 'control request failed'))
        else: fut.set_result(r.get('response') or {})

@patch
async def _handle(self:ClaudeProto, rid, req):
    "Answer one CLI-originated control request; cancelled handlers answer nothing"
    try:
        if req.get('subtype')!='mcp_message': raise ValueError(f"unsupported control request: {req.get('subtype')}")
        r = await mcp_dispatch(req.get('message') or {}, self.schemas, self.ns, self.server)
        if r is None: r = dict(jsonrpc='2.0', result={})
        await self.send(dict(type='control_response', response=dict(subtype='success', request_id=rid, response=dict(mcp_response=r))))
    except asyncio.CancelledError: raise
    except Exception as e: await self.send(dict(type='control_response', response=dict(subtype='error', request_id=rid, error=str(e))))

The read loop ties the pieces together; ending it, from either side, runs `aclose`:

In [ ]:
#| export
@patch
async def events(self:ClaudeProto):
    "Non-control messages from claude, with control traffic routed internally"
    try:
        async for m in read_msgs(self.proc.stdout):
            t = m.get('type')
            if t=='control_response': self._resolve(m)
            elif t=='control_request':
                rid = m.get('request_id')
                task = asyncio.create_task(self._handle(rid, m.get('request') or {}))
                self._inflight[rid] = task
                task.add_done_callback(lambda _,rid=rid: self._inflight.pop(rid, None))
            elif t=='control_cancel_request':
                if task := self._inflight.pop(m.get('request_id'), None): task.cancel()
            else: yield m
    finally: await self.aclose()


In [ ]:
#| export
@patch
async def aclose(self:ClaudeProto):
    "Cancel in-flight handlers and pending requests; the process itself is the runner's to reap"
    for t in list(self._inflight.values()): t.cancel()
    for f in self._pending.values():
        if not f.done(): f.cancel()

A scripted peer exercises the whole contract without spending a model call: it answers our `initialize`, asks for the tool list, calls the first tool, echoes the result back as an assistant message, and finishes. Everything a real claude does, in four lines of traffic:

In [ ]:
# chkstyle: skip
fake = textwrap.dedent('''
    import sys, json
    def w(o): sys.stdout.write(json.dumps(o)+'\\n'); sys.stdout.flush()
    def r(): return json.loads(sys.stdin.readline())
    req = r()
    w(dict(type='control_response', response=dict(subtype='success', request_id=req['request_id'], response=dict(commands=[]))))
    w(dict(type='control_request', request_id='cli_1', request=dict(subtype='mcp_message', server_name='fastclaude', message=dict(jsonrpc='2.0', id=1, method='tools/list'))))
    tools = r()['response']['response']['mcp_response']['result']['tools']
    w(dict(type='control_request', request_id='cli_2', request=dict(subtype='mcp_message', server_name='fastclaude', message=dict(jsonrpc='2.0', id=2, method='tools/call', params=dict(name=tools[0]['name'], arguments=dict(unit='gauss'))))))
    txt = r()['response']['response']['mcp_response']['result']['content'][0]['text']
    w(dict(type='assistant', message=dict(role='assistant', content=[dict(type='text', text=txt)])))
    w(dict(type='result', subtype='success'))
    ''')

Driving it: the events consumer runs as its own task, since `initialize`'s response arrives through the same read loop. The assistant text coming back as our tool's own output proves the full round trip through the bridge:

In [ ]:
p = await asyncio.create_subprocess_exec(sys.executable, '-c', fake, stdin=asyncio.subprocess.PIPE, stdout=asyncio.subprocess.PIPE)
proto = ClaudeProto(p, tools=[flux_meter])
out = []
async def drain():
    async for m in proto.events(): out.append(m)
t = asyncio.create_task(drain())
init = await proto.initialize()
await t
await p.wait()

The handshake response came back matched by id, and the tool's own output arrived as the assistant's text:

In [ ]:
test_eq(init, dict(commands=[]))
test_eq([m['type'] for m in out], ['assistant','result'])
out[0]['message']['content'][0]['text']

When Claude abandons a request it made - the user pressed escape mid-tool, say - it sends `control_cancel_request`. The in-flight handler task is cancelled, the tool sees an ordinary `CancelledError` at its next await, and no response is written for a request nobody is waiting on. An in-memory peer makes the sequencing deterministic:

In [ ]:
class _Sink:
    "Collects written messages; a stand-in for a process stdin"
    def __init__(self): self.msgs = []
    def write(self, b): self.msgs.append(json.loads(b))
    async def drain(self): pass

In [ ]:
hit = []
async def slow():
    "Sleep for a long time."
    try: await asyncio.sleep(30)
    except asyncio.CancelledError:
        hit.append(True)
        raise

Feeding the reader by hand makes the ordering exact: the call starts, parks on its sleep, and only then is cancelled. The tool's own `except` proves the cancellation reached it, and the sink stays empty:

In [ ]:
r2 = asyncio.StreamReader()
fp = AttrDict(stdout=r2, stdin=_Sink())
proto2 = ClaudeProto(fp, tools=[slow])
out2 = []
async def drain2():
    async for m in proto2.events(): out2.append(m)
t2 = asyncio.create_task(drain2())
def feed(o): r2.feed_data(json.dumps(o).encode()+b'\n')

One request goes in flight, then its cancellation, then a normal message to show routing carries on:

In [ ]:
feed(dict(type='control_request', request_id='c1', request=dict(subtype='mcp_message', server_name='fastclaude', message=tcall('slow'))))
await asyncio.sleep(0.05)
feed(dict(type='control_cancel_request', request_id='c1'))
await asyncio.sleep(0.05)
feed(dict(type='result', subtype='success'))
r2.feed_eof()
await t2
test_eq(hit, [True])
test_eq([m['type'] for m in out2], ['result'])
test_eq(fp.stdin.msgs, [])

## A live handshake

The proof that the peer speaks the real protocol is a real process: spawn `claude` with piped stream-json, shake hands, send one user turn, and read to the terminal result. This is the smallest live exchange - no tools, no resume - and the runner in `core` builds on exactly this sequence. It spends tokens, so it stays out of automated runs:

In [ ]:
#| eval: false
td = tempfile.mkdtemp()
argv = ['claude','--output-format','stream-json','--input-format','stream-json','--verbose','--include-partial-messages','--model','sonnet']
lp = await asyncio.create_subprocess_exec(*argv, stdin=asyncio.subprocess.PIPE, stdout=asyncio.subprocess.PIPE, limit=2**25, cwd=td)
lproto = ClaudeProto(lp)
res = []
async def ldrain():
    async for m in lproto.events():
        if m.get('type')=='result': return res.append(m)
lt = asyncio.create_task(ldrain())
(await lproto.initialize()).get('output_style')

In [ ]:
#| eval: false
await lproto.send(dict(type='user', message=dict(role='user', content='Reply with exactly: protocol ok')))
await lt
test('protocol ok', res[0]['result'], in_)
res[0]['result']

In [ ]:
#| eval: false
lp.stdin.close()
await lp.wait()
shutil.rmtree(sess_dir(td), ignore_errors=True)

In [ ]:
#| hide
#| eval: false
import nbdev; nbdev.nbdev_export()